# 05J — Manuscript Results Assembly

This notebook converts the frozen 05C–05I-B scientific outputs into a traceable manuscript-results package.

It generates:

- manuscript-ready primary and supplementary tables;
- a numbered figure package and complete captions;
- draft Methods, Results, and Limitations sections;
- a claim-to-artifact evidence ledger;
- a reproducibility manifest and manuscript handoff.

Scientific guardrails:

- No model is fitted, tuned, recalibrated, or thresholded.
- Performance claims come from incident-held-out OOF or forward-year evaluation.
- The 05F full-data model is identified only as a deployment artifact.
- Interpretations and input regimes are predictive, not causal.
- This package contains project results; literature citations and journal-specific formatting remain a later step.


## 0. Configuration and required frozen artifacts


In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "05J_V1_MANUSCRIPT_RESULTS_ASSEMBLY"

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

PRIMARY_ROOT = PROJECT_ROOT / "outputs" / "primary_model_freeze_v1"
CALIBRATION_ROOT = PROJECT_ROOT / "outputs" / "fixed_primary_nested_calibration_v1"
FINAL_MODEL_ROOT = PROJECT_ROOT / "outputs" / "final_primary_model_v1"
PROXY_ROOT = PROJECT_ROOT / "outputs" / "observation_process_proxy_sensitivity_v1"
TEMPORAL_ROOT = PROJECT_ROOT / "outputs" / "forward_temporal_generalization_v1"
INTERPRETATION_ROOT = PROJECT_ROOT / "outputs" / "frozen_model_interpretation_v1"
REGIME_ROOT = PROJECT_ROOT / "outputs" / "input_regimes_v1"

PRIMARY_POINT_PATH = PRIMARY_ROOT / "PRIMARY_MODEL_POINT_METRICS_V1.csv"
PRIMARY_CI_PATH = PRIMARY_ROOT / "INCIDENT_CLUSTER_BOOTSTRAP_CI_V1.csv"
PRIMARY_YEAR_PATH = PRIMARY_ROOT / "PER_YEAR_METRICS_V1.csv"
PRIMARY_FOLD_PATH = PRIMARY_ROOT / "PER_OUTER_FOLD_METRICS_V1.csv"
PRIMARY_CHECKS_PATH = PRIMARY_ROOT / "FINAL_PRIMARY_MODEL_FREEZE_CHECKS_V1.csv"

CALIBRATION_METRICS_PATH = CALIBRATION_ROOT / "FIXED_PRIMARY_RAW_VS_NESTED_CALIBRATED_METRICS_V1.csv"
CALIBRATION_EFFECT_PATH = CALIBRATION_ROOT / "FIXED_PRIMARY_CALIBRATION_EFFECT_V1.csv"
CALIBRATION_BOOTSTRAP_PATH = CALIBRATION_ROOT / "FIXED_PRIMARY_NESTED_CALIBRATION_BOOTSTRAP_V1.csv"
CALIBRATION_CHECKS_PATH = CALIBRATION_ROOT / "FINAL_FIXED_PRIMARY_CALIBRATION_CHECKS_V1.csv"

MODEL_MANIFEST_PATH = FINAL_MODEL_ROOT / "MODEL_ARTIFACT_MANIFEST_V1.csv"
MODEL_CHECKS_PATH = FINAL_MODEL_ROOT / "FINAL_MODEL_SERIALIZATION_CHECKS_V1.csv"
MODEL_PARAMS_PATH = PROJECT_ROOT / "models" / "fireline_hold_primary_v1" / "fireline_hold_primary_v1_params.json"

PROXY_METRICS_PATH = PROXY_ROOT / "OBSERVATION_PROCESS_SENSITIVITY_METRICS_V1.csv"
PROXY_BOOTSTRAP_PATH = PROXY_ROOT / "OBSERVATION_PROCESS_SENSITIVITY_BOOTSTRAP_V1.csv"
PROXY_CHECKS_PATH = PROXY_ROOT / "FINAL_OBSERVATION_PROCESS_PROXY_CHECKS_V1.csv"

TEMPORAL_METRICS_PATH = INTERPRETATION_ROOT / "TEMPORAL_METRICS_PUBLICATION_CLEAN_V1.csv"
TEMPORAL_AGGREGATE_PATH = INTERPRETATION_ROOT / "ROLLING_ORIGIN_AGGREGATE_METRICS_V1.csv"
TEMPORAL_BOOTSTRAP_PATH = TEMPORAL_ROOT / "FORWARD_TEMPORAL_PAIRED_BOOTSTRAP_V1.csv"
TEMPORAL_CHECKS_PATH = TEMPORAL_ROOT / "FINAL_FORWARD_TEMPORAL_CHECKS_V1.csv"

SOURCE_IMPORTANCE_PATH = INTERPRETATION_ROOT / "OOF_SHAP_SOURCE_GROUP_IMPORTANCE_V1.csv"
FAMILY_IMPORTANCE_PATH = INTERPRETATION_ROOT / "OOF_SHAP_FEATURE_FAMILY_IMPORTANCE_V1.csv"
FEATURE_IMPORTANCE_PATH = INTERPRETATION_ROOT / "OOF_SHAP_FEATURE_IMPORTANCE_V1.csv"
INTERPRETATION_CHECKS_PATH = INTERPRETATION_ROOT / "FINAL_FROZEN_MODEL_INTERPRETATION_CHECKS_V1.csv"

PHYSICAL_REGIMES_PATH = REGIME_ROOT / "PHYSICAL_INPUT_REGIMES_V1.csv"
ALL_INPUT_REGIMES_PATH = REGIME_ROOT / "ALL_INPUT_SENSITIVITY_REGIMES_V1.csv"
REGIME_FIDELITY_PATH = REGIME_ROOT / "CROSSFIT_SURROGATE_FIDELITY_V1.csv"
REGIME_CHECKS_PATH = REGIME_ROOT / "FINAL_INPUT_REGIME_CHECKS_V1.csv"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "manuscript_assembly_v1"
TABLE_ROOT = OUTPUT_ROOT / "tables"
TEXT_ROOT = OUTPUT_ROOT / "text"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
MAIN_FIGURE_ROOT = FIGURE_ROOT / "main"
SUPPLEMENT_FIGURE_ROOT = FIGURE_ROOT / "supplement"
for p in [OUTPUT_ROOT, TABLE_ROOT, TEXT_ROOT, MAIN_FIGURE_ROOT, SUPPLEMENT_FIGURE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

required_paths = [
    PRIMARY_POINT_PATH, PRIMARY_CI_PATH, PRIMARY_YEAR_PATH, PRIMARY_FOLD_PATH, PRIMARY_CHECKS_PATH,
    CALIBRATION_METRICS_PATH, CALIBRATION_EFFECT_PATH, CALIBRATION_BOOTSTRAP_PATH, CALIBRATION_CHECKS_PATH,
    MODEL_MANIFEST_PATH, MODEL_CHECKS_PATH, MODEL_PARAMS_PATH,
    PROXY_METRICS_PATH, PROXY_BOOTSTRAP_PATH, PROXY_CHECKS_PATH,
    TEMPORAL_METRICS_PATH, TEMPORAL_AGGREGATE_PATH, TEMPORAL_BOOTSTRAP_PATH, TEMPORAL_CHECKS_PATH,
    SOURCE_IMPORTANCE_PATH, FAMILY_IMPORTANCE_PATH, FEATURE_IMPORTANCE_PATH, INTERPRETATION_CHECKS_PATH,
    PHYSICAL_REGIMES_PATH, ALL_INPUT_REGIMES_PATH, REGIME_FIDELITY_PATH, REGIME_CHECKS_PATH,
]
for p in required_paths:
    assert p.exists(), p

required = {"pandas": "pandas", "numpy": "numpy"}
for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

from datetime import datetime, timezone
import hashlib
import json
import shutil

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 420)

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381
EXPECTED_FEATURES = 803
EXPECTED_OUTER_FOLDS = 5

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)


## 1. Load and cross-check all frozen result layers


In [ ]:
primary_point = pd.read_csv(PRIMARY_POINT_PATH)
primary_ci = pd.read_csv(PRIMARY_CI_PATH)
primary_year = pd.read_csv(PRIMARY_YEAR_PATH)
primary_fold = pd.read_csv(PRIMARY_FOLD_PATH)

calibration_metrics = pd.read_csv(CALIBRATION_METRICS_PATH)
calibration_effect = pd.read_csv(CALIBRATION_EFFECT_PATH)
calibration_bootstrap = pd.read_csv(CALIBRATION_BOOTSTRAP_PATH)

model_manifest = pd.read_csv(MODEL_MANIFEST_PATH)
with MODEL_PARAMS_PATH.open("r", encoding="utf-8") as f:
    model_params = json.load(f)

proxy_metrics = pd.read_csv(PROXY_METRICS_PATH)
proxy_bootstrap = pd.read_csv(PROXY_BOOTSTRAP_PATH)

temporal_metrics = pd.read_csv(TEMPORAL_METRICS_PATH)
temporal_aggregate = pd.read_csv(TEMPORAL_AGGREGATE_PATH)
temporal_bootstrap = pd.read_csv(TEMPORAL_BOOTSTRAP_PATH)

source_importance = pd.read_csv(SOURCE_IMPORTANCE_PATH)
family_importance = pd.read_csv(FAMILY_IMPORTANCE_PATH)
feature_importance = pd.read_csv(FEATURE_IMPORTANCE_PATH)

physical_regimes = pd.read_csv(PHYSICAL_REGIMES_PATH)
all_input_regimes = pd.read_csv(ALL_INPUT_REGIMES_PATH)
regime_fidelity = pd.read_csv(REGIME_FIDELITY_PATH)

check_paths = {
    "05E_primary_freeze": PRIMARY_CHECKS_PATH,
    "05E1_calibration": CALIBRATION_CHECKS_PATH,
    "05F_serialization": MODEL_CHECKS_PATH,
    "05G_proxy_sensitivity": PROXY_CHECKS_PATH,
    "05H_forward_temporal": TEMPORAL_CHECKS_PATH,
    "05I_interpretation": INTERPRETATION_CHECKS_PATH,
    "05IB_regimes": REGIME_CHECKS_PATH,
}
upstream_check_rows = []
for stage, path in check_paths.items():
    checks_here = pd.read_csv(path)
    assert {"check", "pass"}.issubset(checks_here.columns)
    for _, row in checks_here.iterrows():
        upstream_check_rows.append({"stage": stage, "check": row["check"], "pass": bool(row["pass"])})
upstream_checks = pd.DataFrame(upstream_check_rows)

raw = calibration_metrics.set_index("probability_type").loc["raw"]
calibrated = calibration_metrics.set_index("probability_type").loc["nested_calibrated"]
primary = primary_point.set_index("model_id").loc["05C_FIXED_ROW_RAW"]

assert upstream_checks["pass"].all()
assert int(raw["units"]) == EXPECTED_UNITS
assert int(raw["incidents"]) == EXPECTED_INCIDENTS
assert len(feature_importance) == EXPECTED_FEATURES
assert primary_fold[primary_fold["model_id"] == "05C_FIXED_ROW_RAW"]["outer_fold"].nunique() == EXPECTED_OUTER_FOLDS
for metric in primary.index.intersection(raw.index):
    if pd.api.types.is_number(primary[metric]):
        assert np.isclose(float(primary[metric]), float(raw[metric]), equal_nan=True)
assert np.isclose(source_importance["oof_shap_fraction"].sum(), 1.0)

print("Upstream checks:", len(upstream_checks), "all PASS")
print("Frozen primary units/incidents/features:", EXPECTED_UNITS, EXPECTED_INCIDENTS, EXPECTED_FEATURES)


## 2. Table helpers and manuscript table registry


In [ ]:
def md_table(df):
    try:
        return df.to_markdown(index=False, floatfmt=".4f")
    except Exception:
        return df.to_string(index=False)


table_registry_rows = []


def save_table(df, stem, title, manuscript_role):
    csv_path = TABLE_ROOT / f"{stem}.csv"
    md_path = TABLE_ROOT / f"{stem}.md"
    df.to_csv(csv_path, index=False)
    md_path.write_text(f"# {title}\n\n" + md_table(df) + "\n", encoding="utf-8")
    table_registry_rows.append({
        "table_stem": stem,
        "title": title,
        "manuscript_role": manuscript_role,
        "rows": int(len(df)),
        "csv_path": str(csv_path),
        "markdown_path": str(md_path),
    })
    return df


def metric_label(metric):
    return {
        "mean_pred": "Mean predicted probability",
        "pooled_pr_auc": "Pooled PR-AUC",
        "pooled_roc_auc": "Pooled ROC-AUC",
        "pooled_brier": "Pooled Brier score",
        "pooled_log_loss": "Pooled log loss",
        "incident_macro_brier": "Incident-macro Brier score",
        "incident_macro_log_loss": "Incident-macro log loss",
        "mixed_incident_macro_pr_auc": "Mixed-incident macro PR-AUC",
        "mixed_incident_macro_roc_auc": "Mixed-incident macro ROC-AUC",
    }.get(metric, metric)


## 3. Primary manuscript tables


In [ ]:
source_counts = feature_importance.groupby("source_group")["feature"].size().to_dict()
cohort_contract = pd.DataFrame([
    {"item": "Analysis period", "value": "2020–2024", "scientific_role": "Frozen modeled cohort"},
    {"item": "Analysis units", "value": f"{EXPECTED_UNITS:,}", "scientific_role": "Local fireline units"},
    {"item": "Wildfire incidents", "value": f"{EXPECTED_INCIDENTS:,}", "scientific_role": "Grouping and resampling unit"},
    {"item": "Unit-level Held prevalence", "value": f"{float(raw['held_rate']):.3f}", "scientific_role": "Binary target prevalence"},
    {"item": "Outer validation folds", "value": str(EXPECTED_OUTER_FOLDS), "scientific_role": "Incident-disjoint OOF evaluation"},
    {"item": "Frozen predictors", "value": f"{EXPECTED_FEATURES:,}", "scientific_role": "Primary feature contract"},
    {"item": "Fire-approach predictors", "value": str(source_counts["fire_approach_physical"]), "scientific_role": "Physical/dynamic"},
    {"item": "Weather predictors", "value": str(source_counts["weather"]), "scientific_role": "Physical/dynamic"},
    {"item": "Terrain predictors", "value": str(source_counts["terrain"]), "scientific_role": "Physical/static"},
    {"item": "Fuels/canopy predictors", "value": str(source_counts["fuels_canopy"]), "scientific_role": "Physical/static by retrospective vintage"},
    {"item": "Observation-process predictors", "value": str(source_counts["observation_process"]), "scientific_role": "Isolated in sensitivity analysis"},
    {"item": "Primary estimator", "value": "LightGBM binary classifier", "scientific_role": "Frozen fixed-parameter model"},
    {"item": "Scientific probability output", "value": "Raw P(Held)", "scientific_role": "No post-hoc calibration or threshold"},
])
save_table(cohort_contract, "TABLE_1_COHORT_AND_MODEL_CONTRACT_V1", "Table 1. Cohort and frozen model contract", "main")

metric_order = [
    "mean_pred", "pooled_pr_auc", "pooled_roc_auc", "pooled_brier", "pooled_log_loss",
    "incident_macro_brier", "incident_macro_log_loss", "mixed_incident_macro_pr_auc", "mixed_incident_macro_roc_auc",
]
primary_ci_lookup = primary_ci[primary_ci["model_id"] == "05C_FIXED_ROW_RAW"].set_index("metric")
performance_rows = []
for metric in metric_order:
    ci_low = float(primary_ci_lookup.loc[metric, "ci_low"]) if metric in primary_ci_lookup.index else np.nan
    ci_high = float(primary_ci_lookup.loc[metric, "ci_high"]) if metric in primary_ci_lookup.index else np.nan
    performance_rows.append({
        "metric": metric_label(metric),
        "metric_id": metric,
        "point_estimate": float(primary[metric]),
        "incident_cluster_bootstrap_ci_low": ci_low,
        "incident_cluster_bootstrap_ci_high": ci_high,
        "evaluation_scope": "mixed incidents, macro" if metric.startswith("mixed_incident") else ("incident macro" if metric.startswith("incident_macro") else "pooled units"),
    })
primary_performance = pd.DataFrame(performance_rows)
save_table(primary_performance, "TABLE_2_PRIMARY_OOF_PERFORMANCE_V1", "Table 2. Frozen primary incident-held-out performance", "main")

calibration_table = calibration_effect.copy()
calibration_table["interpretation"] = np.where(
    calibration_table["metric"].str.startswith("incident_macro"),
    "Positive delta favors raw probabilities for lower-is-better incident loss",
    "Calibration effect; see paired bootstrap interval",
)
cal_boot = calibration_bootstrap.rename(columns={"mean_delta": "bootstrap_mean_cal_minus_raw"})
calibration_table = calibration_table.merge(
    cal_boot[["metric", "bootstrap_mean_cal_minus_raw", "ci_low", "ci_high", "probability_calibration_better"]],
    on="metric", how="left", validate="one_to_one",
)
save_table(calibration_table, "TABLE_3_CALIBRATION_DECISION_V1", "Table 3. Raw versus nested-calibrated probability audit", "main")

proxy_table = proxy_metrics.copy()
proxy_table["model_role"] = proxy_table["experiment_id"].map({
    "PRIMARY_FULL": "Frozen primary",
    "PHYSICAL_SENSITIVITY": "Excludes 11 observation-process predictors",
    "PROCESS_ONLY": "Observation-process predictors only",
})
proxy_table = proxy_table[["experiment_id", "model_role", "features", "units", "incidents", "pr_auc", "roc_auc", "brier", "log_loss", "incident_macro_brier", "incident_macro_log_loss", "mixed_macro_pr_auc", "mixed_macro_roc_auc"]]
save_table(proxy_table, "TABLE_4_OBSERVATION_PROCESS_SENSITIVITY_V1", "Table 4. Observation-process sensitivity", "main")

temporal_table = temporal_metrics[[
    "test_year", "model_id", "units", "incidents", "held_rate", "mean_pred", "prediction_bias",
    "pooled_pr_auc", "pooled_roc_auc", "incident_macro_brier", "incident_macro_log_loss",
    "mixed_incident_macro_pr_auc", "mixed_incident_macro_roc_auc", "calibration_estimable",
]].sort_values(["test_year", "model_id"])
save_table(temporal_table, "TABLE_5_FORWARD_TEMPORAL_GENERALIZATION_V1", "Table 5. Rolling-origin forward temporal generalization", "main")

source_table = source_importance[["source_group", "features", "oof_shap_fraction", "total_full_data_gain_fraction"]].copy()
source_table = source_table.sort_values("oof_shap_fraction", ascending=False)
save_table(source_table, "TABLE_6_PREDICTOR_SOURCE_IMPORTANCE_V1", "Table 6. Held-out predictive importance by source", "main")

top_feature_table = feature_importance.head(20)[[
    "rank_oof_shap", "feature", "source_group", "feature_family",
    "mean_abs_oof_shap_incident_balanced", "oof_shap_fraction", "fold_rank_mean", "fold_rank_sd", "top_20_fold_count",
]].copy()
save_table(top_feature_table, "TABLE_7_TOP_HELD_OUT_FEATURES_V1", "Table 7. Leading held-out predictive features", "supplement")

regime_table = physical_regimes[[
    "regime_id", "associated_side", "rule", "units", "incidents", "incident_balanced_held_rate",
    "incident_balanced_held_rate_ci_lower", "incident_balanced_held_rate_ci_upper",
    "incident_balanced_mean_primary_oof_probability", "delta_held_rate_vs_study_incident_mean",
]].copy()
save_table(regime_table, "TABLE_8_PHYSICAL_INPUT_REGIMES_V1", "Table 8. Descriptive physical-input regimes", "main")

save_table(primary_year, "TABLE_S1_PRIMARY_PER_YEAR_METRICS_V1", "Supplementary Table S1. Primary model metrics by year", "supplement")
save_table(primary_fold, "TABLE_S2_PRIMARY_PER_OUTER_FOLD_METRICS_V1", "Supplementary Table S2. Primary model metrics by outer fold", "supplement")
save_table(temporal_aggregate, "TABLE_S3_ROLLING_ORIGIN_AGGREGATE_METRICS_V1", "Supplementary Table S3. Aggregate rolling-origin metrics", "supplement")
save_table(proxy_bootstrap, "TABLE_S4_PROXY_PAIRED_BOOTSTRAP_V1", "Supplementary Table S4. Observation-process paired bootstrap", "supplement")
save_table(temporal_bootstrap, "TABLE_S5_TEMPORAL_PAIRED_BOOTSTRAP_V1", "Supplementary Table S5. Forward temporal paired bootstrap", "supplement")
save_table(family_importance, "TABLE_S6_FEATURE_FAMILY_IMPORTANCE_V1", "Supplementary Table S6. Feature-family held-out importance", "supplement")
save_table(all_input_regimes, "TABLE_S7_ALL_INPUT_REGIMES_V1", "Supplementary Table S7. Observation-process-inclusive regimes", "supplement")
save_table(regime_fidelity, "TABLE_S8_REGIME_SURROGATE_FIDELITY_V1", "Supplementary Table S8. Cross-fold regime-surrogate fidelity", "supplement")

table_registry = pd.DataFrame(table_registry_rows)
table_registry.to_csv(OUTPUT_ROOT / "MANUSCRIPT_TABLE_REGISTRY_V1.csv", index=False)
display(table_registry)


## 4. Numbered figure package and captions


In [ ]:
figure_specs = [
    {
        "figure_id": "Figure 1", "tier": "main", "output_stem": "FIGURE_1_FROZEN_MODEL_ROBUSTNESS_SYNTHESIS_V1",
        "source_dir": INTERPRETATION_ROOT / "figures", "source_stem": "FIG_05I_05_ROBUSTNESS_SYNTHESIS_V1",
        "title": "Frozen-model robustness synthesis",
        "caption": "Frozen-model robustness synthesis. Top panels compare the frozen primary model with the physical-only and observation-process-only sensitivity models using pooled ROC-AUC and incident-macro Brier score. Bottom panels show rolling-origin forward-year mixed-incident macro ROC-AUC and incident-macro Brier score for the fixed 05C model, a historical-prevalence prior, and the mixed-year OOF reference. Lower Brier score indicates better probability accuracy.",
    },
    {
        "figure_id": "Figure 2", "tier": "main", "output_stem": "FIGURE_2_PREDICTOR_SOURCE_IMPORTANCE_V1",
        "source_dir": INTERPRETATION_ROOT / "figures", "source_stem": "FIG_05I_01_SOURCE_GROUP_OOF_SHAP_V1",
        "title": "Held-out predictive importance by source",
        "caption": "Incident-balanced held-out TreeSHAP importance aggregated by predictor source. Values are fractions of mean absolute out-of-fold SHAP contribution on the log-odds scale. Attributions describe the frozen model and are not causal effects or independent modality contributions.",
    },
    {
        "figure_id": "Figure 3", "tier": "main", "output_stem": "FIGURE_3_HELD_OUT_SHAP_SUMMARY_V1",
        "source_dir": INTERPRETATION_ROOT / "figures", "source_stem": "FIG_05I_03_OOF_SHAP_SUMMARY_V1",
        "title": "Held-out SHAP distributions for leading predictors",
        "caption": "Held-out TreeSHAP distributions for leading predictors. Each point represents an incident-balanced sampled fireline unit evaluated by a model that did not train on its incident. Horizontal position is the SHAP contribution to the predicted log odds of Held; color indicates the within-feature value percentile. Correlated predictors can share or redistribute attribution.",
    },
    {
        "figure_id": "Figure 4", "tier": "main", "output_stem": "FIGURE_4_PHYSICAL_INPUT_REGIMES_V1",
        "source_dir": REGIME_ROOT / "figures", "source_stem": "FIG_05IB_02_HELD_RATE_BY_PHYSICAL_REGIME_V1",
        "title": "Held versus Burned-Over associations across physical-input regimes",
        "caption": "Incident-balanced observed Held rates across eight descriptive physical-input regimes, with 95% wildfire-incident bootstrap confidence intervals. The dashed line is the overall incident-balanced Held rate. Regimes are ordered summaries of a depth-three surrogate for frozen OOF probabilities; cut points are descriptive and not operational thresholds.",
    },
    {
        "figure_id": "Figure S1", "tier": "supplement", "output_stem": "FIGURE_S1_FORWARD_TEMPORAL_METRICS_V1",
        "source_dir": TEMPORAL_ROOT / "figures", "source_stem": "FORWARD_TEMPORAL_METRICS_V1",
        "title": "Forward temporal performance by test year",
        "caption": "Rolling-origin forward-year performance for the fixed 05C model, historical-prevalence prior, and mixed-year OOF reference. Each test year was evaluated using only earlier years for the forward model.",
    },
    {
        "figure_id": "Figure S2", "tier": "supplement", "output_stem": "FIGURE_S2_FORWARD_TEMPORAL_RELIABILITY_V1",
        "source_dir": TEMPORAL_ROOT / "figures", "source_stem": "FORWARD_TEMPORAL_RELIABILITY_V1",
        "title": "Forward temporal reliability",
        "caption": "Reliability of rolling-origin forward-year predictions by test year. Departures from the diagonal indicate temporal calibration shift; no post-hoc recalibration was fitted in the frozen primary pipeline.",
    },
    {
        "figure_id": "Figure S3", "tier": "supplement", "output_stem": "FIGURE_S3_TOP_HELD_OUT_FEATURES_V1",
        "source_dir": INTERPRETATION_ROOT / "figures", "source_stem": "FIG_05I_02_TOP_FEATURE_OOF_SHAP_V1",
        "title": "Top held-out predictive features",
        "caption": "Leading individual predictors ranked by incident-balanced mean absolute out-of-fold TreeSHAP contribution. Bar color identifies predictor source.",
    },
    {
        "figure_id": "Figure S4", "tier": "supplement", "output_stem": "FIGURE_S4_FEATURE_RANK_STABILITY_V1",
        "source_dir": INTERPRETATION_ROOT / "figures", "source_stem": "FIG_05I_04_FEATURE_RANK_STABILITY_V1",
        "title": "Outer-fold feature-rank stability",
        "caption": "Within-fold held-out SHAP importance ranks for the twenty leading global features. Lower ranks indicate stronger predictive contribution; variability indicates incident-set heterogeneity.",
    },
    {
        "figure_id": "Figure S5", "tier": "supplement", "output_stem": "FIGURE_S5_GAIN_VS_OOF_SHAP_V1",
        "source_dir": INTERPRETATION_ROOT / "figures", "source_stem": "FIG_05I_06_GAIN_VS_OOF_SHAP_V1",
        "title": "Full-data gain versus held-out SHAP importance",
        "caption": "Comparison of full-data LightGBM gain importance and incident-balanced held-out SHAP importance. Agreement supports broad ranking consistency, while discrepancies emphasize that training gain and held-out attribution answer different questions.",
    },
    {
        "figure_id": "Figure S6", "tier": "supplement", "output_stem": "FIGURE_S6_PHYSICAL_INPUT_REGIME_TREE_V1",
        "source_dir": REGIME_ROOT / "figures", "source_stem": "FIG_05IB_01_PHYSICAL_INPUT_REGIME_TREE_V1",
        "title": "Descriptive physical-input regime tree",
        "caption": "Depth-three incident-balanced regression tree approximating frozen OOF probabilities using a prespecified physical/environmental candidate set. Node values are mean surrogate targets; the tree is interpretive rather than a replacement predictive model.",
    },
    {
        "figure_id": "Figure S7", "tier": "supplement", "output_stem": "FIGURE_S7_REGIME_SURROGATE_FIDELITY_V1",
        "source_dir": REGIME_ROOT / "figures", "source_stem": "FIG_05IB_03_CROSSFIT_SURROGATE_FIDELITY_V1",
        "title": "Cross-fold fidelity of shallow regime trees",
        "caption": "Binned agreement between cross-fitted shallow-surrogate probabilities and frozen primary OOF probabilities for physical-only and observation-process-inclusive candidate sets. Moderate fidelity indicates that regimes summarize, but do not reproduce, the full model.",
    },
    {
        "figure_id": "Figure S8", "tier": "supplement", "output_stem": "FIGURE_S8_PHYSICAL_SPLIT_STABILITY_V1",
        "source_dir": REGIME_ROOT / "figures", "source_stem": "FIG_05IB_04_PHYSICAL_SPLIT_STABILITY_V1",
        "title": "Physical regime split stability",
        "caption": "Number of outer folds in which each physical predictor was selected by a depth-three surrogate tree. Recurrent selection supports qualitative regime structure but does not validate universal cut points.",
    },
]

figure_rows = []
for spec in figure_specs:
    destination_root = MAIN_FIGURE_ROOT if spec["tier"] == "main" else SUPPLEMENT_FIGURE_ROOT
    paths = {}
    for extension in ["png", "pdf"]:
        source = spec["source_dir"] / f"{spec['source_stem']}.{extension}"
        destination = destination_root / f"{spec['output_stem']}.{extension}"
        assert source.exists(), source
        shutil.copy2(source, destination)
        paths[extension] = destination
    figure_rows.append({
        "figure_id": spec["figure_id"],
        "tier": spec["tier"],
        "title": spec["title"],
        "caption": spec["caption"],
        "source_stem": spec["source_stem"],
        "manuscript_png": str(paths["png"]),
        "manuscript_pdf": str(paths["pdf"]),
    })

figure_inventory = pd.DataFrame(figure_rows)
figure_inventory.to_csv(OUTPUT_ROOT / "MANUSCRIPT_FIGURE_INVENTORY_V1.csv", index=False)
caption_lines = ["# Manuscript Figure Captions", ""]
for row in figure_rows:
    caption_lines.extend([f"## {row['figure_id']}. {row['title']}", "", row["caption"], ""])
(TEXT_ROOT / "FIGURE_CAPTIONS_V1.md").write_text("\n".join(caption_lines), encoding="utf-8")
display(figure_inventory[["figure_id", "tier", "title", "source_stem"]])


## 5. Draft Methods, Results, and Limitations


In [ ]:
def f3(value):
    return f"{float(value):.3f}"


def ci_for(metric):
    row = primary_ci[(primary_ci["model_id"] == "05C_FIXED_ROW_RAW") & (primary_ci["metric"] == metric)].iloc[0]
    return float(row["ci_low"]), float(row["ci_high"])


macro_brier_ci = ci_for("incident_macro_brier")
macro_log_ci = ci_for("incident_macro_log_loss")
macro_pr_ci = ci_for("mixed_incident_macro_pr_auc")
macro_roc_ci = ci_for("mixed_incident_macro_roc_auc")

proxy_indexed = proxy_metrics.set_index("experiment_id")
forward_agg = temporal_aggregate.set_index("model_id").loc["FORWARD_FIXED_05C"]
prior_agg = temporal_aggregate.set_index("model_id").loc["HISTORICAL_PREVALENCE_PRIOR"]
reference_agg = temporal_aggregate.set_index("model_id").loc["MIXED_YEAR_PRIMARY_OOF_REFERENCE"]

weather_fraction = float(source_importance.set_index("source_group").loc["weather", "oof_shap_fraction"])
fuel_fraction = float(source_importance.set_index("source_group").loc["fuels_canopy", "oof_shap_fraction"])
process_fraction = float(source_importance.set_index("source_group").loc["observation_process", "oof_shap_fraction"])

physical_fidelity = regime_fidelity[regime_fidelity["analysis_mode"] == "PHYSICAL_ONLY"]
all_fidelity = regime_fidelity[regime_fidelity["analysis_mode"] == "ALL_INPUT_SENSITIVITY"]
lowest_regime = physical_regimes.sort_values("incident_balanced_held_rate").iloc[0]
highest_regime = physical_regimes.sort_values("incident_balanced_held_rate").iloc[-1]
regime_counts = physical_regimes["associated_side"].value_counts().to_dict()

methods_text = f"""# Draft Methods

## Outcome and analytical cohort

The frozen analytical cohort contained {EXPECTED_UNITS:,} local fireline units from {EXPECTED_INCIDENTS} wildfire incidents during 2020–2024. The binary outcome distinguished Held (`hold_label = 1`) from Burned Over (`hold_label = 0`) units. A unit was eligible when weighted temporal support was at least 0.10. Held units additionally required zero weighted conflict. The target definition, spatial representation, and temporal assignment were frozen before model comparison.

## Predictors and validation design

The frozen primary feature contract contained {EXPECTED_FEATURES} predictors: {source_counts['fire_approach_physical']} fire-approach, {source_counts['weather']} weather, {source_counts['terrain']} terrain, {source_counts['fuels_canopy']} fuels/canopy, and {source_counts['observation_process']} observation-process variables. Wildfire incident was the grouping unit for five-fold outer validation, so all units from an incident remained in one fold. Performance estimates reported for the scientific model are out-of-fold predictions for incidents excluded from model fitting.

## Primary model and uncertainty

The primary model was a fixed-parameter LightGBM binary classifier with {model_params['n_estimators']} trees, learning rate {model_params['learning_rate']}, {model_params['num_leaves']} leaves, minimum child sample size {model_params['min_child_samples']}, and column subsampling fraction {model_params['colsample_bytree']}. Candidate tuning and calibration analyses did not displace the fixed raw-probability model under the incident-centered evaluation criteria. Uncertainty was estimated with 5,000 paired wildfire-incident bootstrap replicates. No probability threshold was selected.

## Calibration and robustness analyses

Nested logistic calibration maps were fit without access to outer test incidents and compared with the raw probabilities. Observation-process sensitivity was assessed by refitting the fixed model after excluding all 11 observation-process variables and by fitting a process-only comparator. Forward temporal generalization used rolling-origin splits for test years 2021–2024, with each forward model trained only on earlier years; a historical-prevalence prior and the mixed-year OOF predictions were retained as references.

## Held-out interpretation and input regimes

TreeSHAP contributions were computed from outer-fold models for a sample of at most 50 units per incident, covering all {EXPECTED_INCIDENTS} incidents. Absolute SHAP values were first averaged within incidents and then equally across incidents. A prespecified depth-three regression-tree surrogate summarized combinations of physical/environmental inputs associated with the frozen OOF probabilities. Regime outcome rates and 95% confidence intervals were calculated by resampling wildfire incidents. SHAP effects, surrogate rules, and cut points were treated as noncausal descriptive associations.
"""

results_text = f"""# Draft Results

## Cohort and primary predictive performance

The cohort included {EXPECTED_UNITS:,} fireline units from {EXPECTED_INCIDENTS} incidents, of which {100 * float(raw['held_rate']):.1f}% were labeled Held. The frozen primary model achieved pooled PR-AUC {f3(primary['pooled_pr_auc'])} and ROC-AUC {f3(primary['pooled_roc_auc'])}. Incident-macro Brier score was {f3(primary['incident_macro_brier'])} (95% incident-bootstrap CI {f3(macro_brier_ci[0])}–{f3(macro_brier_ci[1])}), and incident-macro log loss was {f3(primary['incident_macro_log_loss'])} ({f3(macro_log_ci[0])}–{f3(macro_log_ci[1])}). Across incidents containing both outcomes, macro PR-AUC was {f3(primary['mixed_incident_macro_pr_auc'])} ({f3(macro_pr_ci[0])}–{f3(macro_pr_ci[1])}) and macro ROC-AUC was {f3(primary['mixed_incident_macro_roc_auc'])} ({f3(macro_roc_ci[0])}–{f3(macro_roc_ci[1])}).

## Probability calibration decision

Raw predictions averaged {f3(raw['mean_pred'])} compared with an observed unit-level Held rate of {f3(raw['held_rate'])}. Nested calibration aligned the mean prediction ({f3(calibrated['mean_pred'])}) and slightly reduced pooled Brier score from {f3(raw['pooled_brier'])} to {f3(calibrated['pooled_brier'])}. However, it increased incident-macro Brier score from {f3(raw['incident_macro_brier'])} to {f3(calibrated['incident_macro_brier'])}; the paired bootstrap calibration-minus-raw difference was {f3(calibration_bootstrap.set_index('metric').loc['incident_macro_brier', 'mean_delta'])} (95% CI {f3(calibration_bootstrap.set_index('metric').loc['incident_macro_brier', 'ci_low'])}–{f3(calibration_bootstrap.set_index('metric').loc['incident_macro_brier', 'ci_high'])}). The raw probabilities were therefore retained for the incident-centered primary pipeline, with calibration limitations reported explicitly.

## Observation-process robustness

Removing the 11 observation-process variables preserved pooled discrimination (ROC-AUC {f3(proxy_indexed.loc['PHYSICAL_SENSITIVITY', 'roc_auc'])} versus {f3(proxy_indexed.loc['PRIMARY_FULL', 'roc_auc'])} for the primary model), although incident-macro Brier score increased from {f3(proxy_indexed.loc['PRIMARY_FULL', 'incident_macro_brier'])} to {f3(proxy_indexed.loc['PHYSICAL_SENSITIVITY', 'incident_macro_brier'])}. The process-only comparator had ROC-AUC {f3(proxy_indexed.loc['PROCESS_ONLY', 'roc_auc'])} and mixed-incident macro ROC-AUC {f3(proxy_indexed.loc['PROCESS_ONLY', 'mixed_macro_roc_auc'])}. Thus, observation-process variables contributed to probability accuracy but did not account for the model's principal discrimination.

## Forward temporal generalization

Across rolling-origin test years 2021–2024, the fixed forward model achieved pooled ROC-AUC {f3(forward_agg['pooled_roc_auc'])}, mixed-incident macro ROC-AUC {f3(forward_agg['mixed_incident_macro_roc_auc'])}, and incident-macro Brier score {f3(forward_agg['incident_macro_brier'])}. Its incident-macro Brier score was lower than the historical-prevalence prior ({f3(prior_agg['incident_macro_brier'])}) and close to the mixed-year OOF reference ({f3(reference_agg['incident_macro_brier'])}). Performance varied by year, indicating temporal shifts in prevalence and calibration, while useful within-incident discrimination generally persisted.

## Predictor importance

Weather accounted for {100 * weather_fraction:.1f}% of incident-balanced mean absolute held-out SHAP attribution, followed by fuels/canopy ({100 * fuel_fraction:.1f}%), observation-process variables ({100 * process_fraction:.1f}%), terrain ({100 * float(source_importance.set_index('source_group').loc['terrain', 'oof_shap_fraction']):.1f}%), and physical fire-approach variables ({100 * float(source_importance.set_index('source_group').loc['fire_approach_physical', 'oof_shap_fraction']):.1f}%). Leading stable predictors included observation timing, 12–18-hour relative humidity, observation-history weights, gust speed, fuel-model fractions, crown depth, slope, distance, and elevation. These attributions describe conditional use by the fitted model and do not imply causal or stand-alone modality effects.

## Physical input regimes

The shallow physical-input surrogate identified {len(physical_regimes)} regimes: {regime_counts.get('Burned-over-associated', 0)} Burned-Over-associated, {regime_counts.get('Intermediate/uncertain', 0)} intermediate or uncertain, and {regime_counts.get('Held-associated', 0)} Held-associated. Incident-balanced Held rates ranged from {f3(lowest_regime['incident_balanced_held_rate'])} ({f3(lowest_regime['incident_balanced_held_rate_ci_lower'])}–{f3(lowest_regime['incident_balanced_held_rate_ci_upper'])}) to {f3(highest_regime['incident_balanced_held_rate'])} ({f3(highest_regime['incident_balanced_held_rate_ci_lower'])}–{f3(highest_regime['incident_balanced_held_rate_ci_upper'])}). Relative humidity, fire-to-line distance, terrain elevation, canopy crown depth, and gust speed recurred in the regime structure. Mean cross-fold Spearman fidelity was {physical_fidelity['spearman_r'].mean():.3f} for the physical surrogate and {all_fidelity['spearman_r'].mean():.3f} when observation-process inputs were permitted, confirming that these rules are useful summaries rather than replacements for the full model.
"""

limitations_text = """# Draft Limitations

1. **Observational outcome:** Held and Burned Over labels were derived from spatial and temporal overlay evidence. The analysis estimates predictive association, not causal fireline effectiveness or treatment effect.
2. **Outcome uncertainty:** Support and conflict rules reduce ambiguity but cannot eliminate perimeter-timing error, positional error, or uncertainty in the timing and status of individual fireline segments.
3. **Incident and temporal transportability:** Performance varied across outer folds and forward test years. Predictions may require renewed validation under different regions, seasons, operational practices, or future fire-weather distributions.
4. **Calibration:** The raw primary probabilities showed pooled overprediction. Nested calibration improved some pooled measures but worsened incident-macro probability loss, so no universal post-hoc calibration map was adopted.
5. **Correlated predictors:** Weather, fuels, terrain, and approach variables are correlated. SHAP attribution can be distributed among correlated fields and should not be interpreted as an independent effect or causal ranking.
6. **Observation-process information:** Observation-history variables improved probability accuracy. Sensitivity analyses reduced concern that they solely drove discrimination but cannot prove the absence of all measurement-process dependence.
7. **Regime thresholds:** The shallow-tree cut points are descriptive approximations of the frozen OOF model. They are not operational decision thresholds and require independent validation before field use.
8. **Full-data model:** The serialized 05F model is a deployment artifact trained on all available units. It has no independent performance estimate; all scientific performance claims must remain tied to OOF or forward-year results.
9. **Scope of manuscript package:** This notebook assembles internal scientific results. Literature synthesis, external citations, author contributions, data-access language, and journal-specific reporting requirements remain to be completed.
"""

(TEXT_ROOT / "MANUSCRIPT_METHODS_DRAFT_V1.md").write_text(methods_text, encoding="utf-8")
(TEXT_ROOT / "MANUSCRIPT_RESULTS_DRAFT_V1.md").write_text(results_text, encoding="utf-8")
(TEXT_ROOT / "MANUSCRIPT_LIMITATIONS_DRAFT_V1.md").write_text(limitations_text, encoding="utf-8")

combined_text = "\n\n".join([
    "# Frozen Wildfire Fireline Effectiveness Manuscript Results Package",
    methods_text,
    results_text,
    limitations_text,
    (TEXT_ROOT / "FIGURE_CAPTIONS_V1.md").read_text(encoding="utf-8"),
])
(TEXT_ROOT / "MANUSCRIPT_RESULTS_PACKAGE_V1.md").write_text(combined_text, encoding="utf-8")

print(methods_text[:900])
print("...")
print(results_text[:900])


## 6. Claim ledger, reproducibility manifest, and handoff


In [ ]:
claim_rows = [
    {"claim_id": "C01", "claim": "The frozen cohort contains 304,143 units from 381 incidents during 2020–2024.", "evidence": "TABLE_1_COHORT_AND_MODEL_CONTRACT_V1.csv", "allowed_scope": "descriptive", "avoid": "claiming national completeness"},
    {"claim_id": "C02", "claim": f"Primary pooled PR-AUC={primary['pooled_pr_auc']:.3f} and ROC-AUC={primary['pooled_roc_auc']:.3f}.", "evidence": "TABLE_2_PRIMARY_OOF_PERFORMANCE_V1.csv", "allowed_scope": "incident-held-out OOF", "avoid": "calling full-data training performance"},
    {"claim_id": "C03", "claim": f"Primary incident-macro Brier={primary['incident_macro_brier']:.3f} with incident-bootstrap uncertainty.", "evidence": "TABLE_2_PRIMARY_OOF_PERFORMANCE_V1.csv", "allowed_scope": "incident-held-out OOF", "avoid": "unit-independence confidence intervals"},
    {"claim_id": "C04", "claim": "Nested calibration improved small pooled losses but worsened incident-macro probability loss; raw probabilities remained primary.", "evidence": "TABLE_3_CALIBRATION_DECISION_V1.csv", "allowed_scope": "nested calibration audit", "avoid": "claiming raw probabilities are perfectly calibrated"},
    {"claim_id": "C05", "claim": "Physical-only predictors retained nearly all pooled discrimination, whereas process-only discrimination was weak within incidents.", "evidence": "TABLE_4_OBSERVATION_PROCESS_SENSITIVITY_V1.csv; TABLE_S4_PROXY_PAIRED_BOOTSTRAP_V1.csv", "allowed_scope": "fixed-fold sensitivity", "avoid": "claiming proof of no leakage"},
    {"claim_id": "C06", "claim": "The fixed model retained useful signal under rolling-origin forward-year evaluation and outperformed a historical-prevalence prior in probability loss.", "evidence": "TABLE_5_FORWARD_TEMPORAL_GENERALIZATION_V1.csv; TABLE_S5_TEMPORAL_PAIRED_BOOTSTRAP_V1.csv", "allowed_scope": "2021–2024 forward tests", "avoid": "claiming invariant future performance"},
    {"claim_id": "C07", "claim": f"Weather and fuels/canopy contributed {100*weather_fraction:.1f}% and {100*fuel_fraction:.1f}% of held-out SHAP attribution.", "evidence": "TABLE_6_PREDICTOR_SOURCE_IMPORTANCE_V1.csv; Figure 2", "allowed_scope": "conditional model attribution", "avoid": "causal or independent modality effects"},
    {"claim_id": "C08", "claim": "Observation-process variables were important to the fitted model but were not sufficient to reproduce its within-incident discrimination.", "evidence": "TABLE_4_OBSERVATION_PROCESS_SENSITIVITY_V1.csv; Figure 1", "allowed_scope": "predictive sensitivity", "avoid": "claiming measurement bias is impossible"},
    {"claim_id": "C09", "claim": "Eight descriptive physical-input regimes separated lower and higher incident-balanced Held rates.", "evidence": "TABLE_8_PHYSICAL_INPUT_REGIMES_V1.csv; Figure 4", "allowed_scope": "post-freeze surrogate description", "avoid": "operational thresholds"},
    {"claim_id": "C10", "claim": "Relative humidity, distance, elevation, crown depth, and gust speed recurred in shallow regime rules.", "evidence": "TABLE_8_PHYSICAL_INPUT_REGIMES_V1.csv; Figure S8", "allowed_scope": "qualitative recurrence", "avoid": "causal mechanisms without external evidence"},
    {"claim_id": "C11", "claim": "The 05F full-data model was serialized exactly and is a deployment artifact.", "evidence": "MODEL_ARTIFACT_MANIFEST_V1.csv", "allowed_scope": "artifact provenance", "avoid": "performance estimates from the full-data fit"},
]
claim_ledger = pd.DataFrame(claim_rows)
claim_ledger.to_csv(OUTPUT_ROOT / "MANUSCRIPT_CLAIM_LEDGER_V1.csv", index=False)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


manifest_rows = []
for path in required_paths:
    manifest_rows.append({
        "artifact_role": "frozen_input",
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    })
for path in sorted(TABLE_ROOT.glob("*")) + sorted(TEXT_ROOT.glob("*")) + sorted(FIGURE_ROOT.rglob("*")):
    if path.is_file():
        manifest_rows.append({
            "artifact_role": "manuscript_output",
            "path": str(path),
            "size_bytes": int(path.stat().st_size),
            "sha256": sha256_file(path),
        })
reproducibility_manifest = pd.DataFrame(manifest_rows)
reproducibility_manifest.to_csv(OUTPUT_ROOT / "MANUSCRIPT_REPRODUCIBILITY_MANIFEST_V1.csv", index=False)

handoff_text = f"""# Manuscript Assembly Handoff

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Frozen scientific decisions

- Primary scientific model: fixed 05C LightGBM with raw `P(Held)` output.
- Scientific performance source: incident-held-out OOF and rolling-origin forward-year evaluation.
- Full-data 05F model: deployment artifact only.
- Post-hoc calibration: not adopted for the primary pipeline.
- Probability or classification threshold: none selected.
- Interpretability: held-out incident-balanced TreeSHAP, noncausal.
- Physical regimes: descriptive depth-three surrogate rules, not operational thresholds.

## Package contents

- {len(table_registry)} registered tables in `{TABLE_ROOT}`.
- {len(figure_inventory[figure_inventory['tier'] == 'main'])} main figures and {len(figure_inventory[figure_inventory['tier'] == 'supplement'])} supplementary figures in `{FIGURE_ROOT}`.
- Draft Methods, Results, Limitations, and figure captions in `{TEXT_ROOT}`.
- Claim ledger: `{OUTPUT_ROOT / 'MANUSCRIPT_CLAIM_LEDGER_V1.csv'}`.
- Reproducibility manifest: `{OUTPUT_ROOT / 'MANUSCRIPT_REPRODUCIBILITY_MANIFEST_V1.csv'}`.

## Next manuscript work

1. Select a target journal and apply its structure, word count, and figure limits.
2. Conduct a focused literature review for the Introduction and Discussion.
3. Add dataset citations, software citations, author contributions, funding, and data/code availability language.
4. Review terminology with wildfire operations and fire-behavior domain experts.
5. Perform final statistical and cartographic review before submission.
"""
(OUTPUT_ROOT / "MANUSCRIPT_HANDOFF_V1.md").write_text(handoff_text, encoding="utf-8")
display(claim_ledger)


## 7. Integrity checks and assembly report


In [ ]:
expected_text_files = [
    TEXT_ROOT / "MANUSCRIPT_METHODS_DRAFT_V1.md",
    TEXT_ROOT / "MANUSCRIPT_RESULTS_DRAFT_V1.md",
    TEXT_ROOT / "MANUSCRIPT_LIMITATIONS_DRAFT_V1.md",
    TEXT_ROOT / "MANUSCRIPT_RESULTS_PACKAGE_V1.md",
    TEXT_ROOT / "FIGURE_CAPTIONS_V1.md",
]
expected_output_files = [
    OUTPUT_ROOT / "MANUSCRIPT_TABLE_REGISTRY_V1.csv",
    OUTPUT_ROOT / "MANUSCRIPT_FIGURE_INVENTORY_V1.csv",
    OUTPUT_ROOT / "MANUSCRIPT_CLAIM_LEDGER_V1.csv",
    OUTPUT_ROOT / "MANUSCRIPT_REPRODUCIBILITY_MANIFEST_V1.csv",
    OUTPUT_ROOT / "MANUSCRIPT_HANDOFF_V1.md",
]
copied_figure_paths = []
for _, row in figure_inventory.iterrows():
    copied_figure_paths.extend([Path(row["manuscript_png"]), Path(row["manuscript_pdf"])])

cal_boot_indexed = calibration_bootstrap.set_index("metric")
checks = pd.DataFrame([
    {"check": "all_upstream_05e_through_05ib_checks_pass", "pass": bool(upstream_checks["pass"].all())},
    {"check": "exact_frozen_cohort_contract", "pass": int(raw["units"]) == EXPECTED_UNITS and int(raw["incidents"]) == EXPECTED_INCIDENTS},
    {"check": "exact_803_feature_contract", "pass": len(feature_importance) == EXPECTED_FEATURES},
    {"check": "exact_five_incident_outer_folds", "pass": primary_fold[primary_fold["model_id"] == "05C_FIXED_ROW_RAW"]["outer_fold"].nunique() == EXPECTED_OUTER_FOLDS},
    {"check": "raw_primary_matches_frozen_05e_metrics", "pass": np.isclose(float(primary["pooled_roc_auc"]), float(raw["pooled_roc_auc"]))},
    {"check": "incident_macro_calibration_harm_ci_excludes_zero", "pass": float(cal_boot_indexed.loc["incident_macro_brier", "ci_low"]) > 0},
    {"check": "physical_sensitivity_retains_discrimination", "pass": float(proxy_indexed.loc["PHYSICAL_SENSITIVITY", "roc_auc"]) > 0.70},
    {"check": "process_only_mixed_incident_roc_near_chance", "pass": float(proxy_indexed.loc["PROCESS_ONLY", "mixed_macro_roc_auc"]) < 0.55},
    {"check": "exact_2021_through_2024_forward_years", "pass": set(temporal_metrics["test_year"].unique()) == {2021, 2022, 2023, 2024}},
    {"check": "shap_source_fractions_sum_to_one", "pass": np.isclose(source_importance["oof_shap_fraction"].sum(), 1.0)},
    {"check": "regime_statuses_include_both_sides_and_uncertain", "pass": set(physical_regimes["associated_side"]) == {"Held-associated", "Burned-over-associated", "Intermediate/uncertain"}},
    {"check": "all_registered_tables_written_csv_and_markdown", "pass": bool(table_registry[["csv_path", "markdown_path"]].map(lambda p: Path(p).exists()).all().all())},
    {"check": "four_main_and_eight_supplement_figures", "pass": int((figure_inventory["tier"] == "main").sum()) == 4 and int((figure_inventory["tier"] == "supplement").sum()) == 8},
    {"check": "all_figure_png_and_pdf_files_copied", "pass": all(p.exists() for p in copied_figure_paths)},
    {"check": "all_manuscript_text_files_written", "pass": all(p.exists() and p.stat().st_size > 0 for p in expected_text_files)},
    {"check": "all_manuscript_control_files_written", "pass": all(p.exists() and p.stat().st_size > 0 for p in expected_output_files)},
    {"check": "claim_ledger_has_evidence_and_avoid_language", "pass": bool(claim_ledger[["evidence", "avoid"]].notna().all().all())},
    {"check": "full_data_fit_explicitly_deployment_only", "pass": "deployment artifact" in limitations_text and "deployment artifact" in handoff_text},
    {"check": "no_model_fitting_or_tuning_in_05j", "pass": True},
    {"check": "no_calibration_or_threshold_selection_in_05j", "pass": True},
    {"check": "interpretation_and_regimes_marked_noncausal", "pass": "noncausal" in methods_text and "not operational" in limitations_text},
])
PASS = bool(checks["pass"].all())
checks.to_csv(OUTPUT_ROOT / "FINAL_MANUSCRIPT_ASSEMBLY_CHECKS_V1.csv", index=False)
display(checks)

report_lines = [
    "# 05J Manuscript Results Assembly",
    "",
    f"Build: `{NOTEBOOK_BUILD_ID}`",
    "",
    f"Generated UTC: {datetime.now(timezone.utc).isoformat()}",
    "",
    "## Scientific status",
    "",
    "Frozen-results assembly only. No model fitting, tuning, recalibration, or threshold selection occurred.",
    "",
    "All performance statements are traceable to incident-held-out OOF or rolling-origin forward-year results. The full-data model remains a deployment artifact without an independent performance estimate.",
    "",
    "## Package summary",
    "",
    md_table(pd.DataFrame([
        {"component": "registered tables", "count": len(table_registry)},
        {"component": "main figures", "count": int((figure_inventory["tier"] == "main").sum())},
        {"component": "supplementary figures", "count": int((figure_inventory["tier"] == "supplement").sum())},
        {"component": "claim-ledger entries", "count": len(claim_ledger)},
        {"component": "reproducibility manifest files", "count": len(reproducibility_manifest)},
    ])),
    "",
    "## Final checks",
    "",
    md_table(checks),
    "",
    "## Decision",
    "",
    "`PASS / FROZEN RESULTS ASSEMBLED; READY FOR JOURNAL-SPECIFIC MANUSCRIPT DEVELOPMENT`" if PASS else "`REVIEW REQUIRED`",
]
REPORT_PATH = OUTPUT_ROOT / "MANUSCRIPT_ASSEMBLY_REPORT_V1.md"
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

print()
print("05J MANUSCRIPT ASSEMBLY:", "PASS" if PASS else "REVIEW REQUIRED")
print("Saved:", REPORT_PATH)
print("DONE:", NOTEBOOK_BUILD_ID)


## 8. Output inventory


In [ ]:
print("=" * 100)
print("05J OUTPUTS")
print("=" * 100)
for p in sorted(OUTPUT_ROOT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUTPUT_ROOT)!s:<100} {p.stat().st_size/(1024**2):>10.3f} MB")
